In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# 1. The RGB Conversion Transform
# Caltech-101 has some 1-channel images. This forces everything to 3-channel.
class ConvertRGB:
    def __call__(self, img):
        return img.convert('RGB')

# 2. Standardizing the Input Pipeline
transform = transforms.Compose([
    ConvertRGB(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Loading and Splitting the Dataset
print("Downloading and preparing Caltech-101...")
dataset = datasets.Caltech101(root='./data', download=True, transform=transform)

# 80/20 Train-Test Split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
class AdaptedLeNet5(nn.Module):
    def __init__(self, num_classes=101):
        super(AdaptedLeNet5, self).__init__()
        # Input: 3 x 224 x 224
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5) # Out: 6 x 220 x 220
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Out: 6 x 110 x 110
        
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5) # Out: 16 x 106 x 106
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Out: 16 x 53 x 53
        
        self.flatten = nn.Flatten()
        
        # 16 channels * 53 width * 53 height = 44944
        self.fc1 = nn.Linear(16 * 53 * 53, 44944 / 4 )
        self.fc2 = nn.Linear(44944 / 4, 44944 / 64)
        self.fc3 = nn.Linear(44944 / 64, num_classes)

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [3]:
class AlexNetScratch(nn.Module):
    def __init__(self, num_classes=101):
        super(AlexNetScratch, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [4]:
def get_pretrained_resnet(num_classes=101):
    # Load the pretrained model
    weights = models.ResNet18_Weights.DEFAULT
    resnet = models.resnet18(weights=weights)
    
    # Freeze the convolutional backbone to speed up training
    for param in resnet.parameters():
        param.requires_grad = False
        
    # Replace the final fully connected layer
    # The new layer automatically has requires_grad=True
    num_ftrs = resnet.fc.in_features
    resnet.fc = nn.Linear(num_ftrs, num_classes)
    
    return resnet

In [8]:
import copy
from tqdm import tqdm # Provides the Keras-style 'verbose=1' progress bar

def train_and_evaluate(model, name, train_loader, test_loader, epochs=10, patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    # Only optimize parameters that require gradients (important for frozen ResNet)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
    
    print(f"\n{'='*50}\nStarting Training: {name}\n{'='*50}")
    
    # Early Stopping Variables
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        # -------------------------
        # 1. TRAINING PHASE
        # -------------------------
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        # tqdm creates the verbose=1 progress bar
        train_loop = tqdm(train_loader, leave=False, desc=f"Epoch [{epoch+1}/{epochs}] Train")
        
        for images, labels in train_loop:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            # Multiply by batch size for accurate averaging later
            train_loss += loss.item() * images.size(0) 
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            # Update the progress bar dynamically
            current_loss = train_loss / train_total
            current_acc = 100. * train_correct / train_total
            train_loop.set_postfix(loss=f"{current_loss:.4f}", acc=f"{current_acc:.2f}%")
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        avg_train_acc = 100. * train_correct / train_total

        # -------------------------
        # 2. VALIDATION PHASE
        # -------------------------
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad(): # Disable gradient tracking for evaluation
            val_loop = tqdm(test_loader, leave=False, desc=f"Epoch [{epoch+1}/{epochs}] Valid")
            for images, labels in val_loop:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        avg_val_loss = val_loss / len(test_loader.dataset)
        avg_val_acc = 100. * val_correct / val_total
        
        # Print epoch summary
        print(f"Epoch [{epoch+1}/{epochs}] - "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.2f}%")

        # -------------------------
        # 3. EARLY STOPPING LOGIC
        # -------------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # Save the best weights
            best_model_wts = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            print(f"--> Validation loss did not improve. Patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print(f"*** Early Stopping Triggered at Epoch {epoch+1} ***")
                break
                
    # Restore best weights before returning
    model.load_state_dict(best_model_wts)
    print(f"Finished {name}. Best Validation Loss: {best_val_loss:.4f}")
    return model

# ==========================================
# 1. Model Instantiation
# ==========================================
# Instantiating the exact classes and functions defined in Phase 2
lenet = AdaptedLeNet5()
alexnet = AlexNetScratch()
resnet = get_pretrained_resnet()

# ==========================================
# 2. Execute the Benchmark
# ==========================================
# Running the unified training loop with early stopping and accuracy monitoring
lenet_model = train_and_evaluate(
    model=lenet, 
    name="LeNet-5 (Scratch)", 
    train_loader=train_loader, 
    test_loader=test_loader, 
    epochs=10, 
    patience=3
)

alexnet_model = train_and_evaluate(
    model=alexnet, 
    name="AlexNet (Scratch)", 
    train_loader=train_loader, 
    test_loader=test_loader, 
    epochs=10, 
    patience=3
)

resnet_model = train_and_evaluate(
    model=resnet, 
    name="ResNet-18 (Transfer Learning)", 
    train_loader=train_loader, 
    test_loader=test_loader, 
    epochs=10, 
    patience=3
)


Starting Training: LeNet-5 (Scratch)


Epoch [1/10] - Train Loss: 4.0317, Train Acc: 17.92% | Val Loss: 3.4348, Val Acc: 30.18%


Epoch [2/10] - Train Loss: 3.1264, Train Acc: 34.35% | Val Loss: 2.9992, Val Acc: 37.79%


Epoch [3/10] - Train Loss: 2.4973, Train Acc: 44.56% | Val Loss: 2.6489, Val Acc: 43.09%


Epoch [4/10] - Train Loss: 1.9390, Train Acc: 55.37% | Val Loss: 2.6219, Val Acc: 45.33%


...

Epoch [6/10] - Train Loss: 0.3776, Train Acc: 94.90% | Val Loss: 0.4686, Val Acc: 92.17%


Epoch [7/10] - Train Loss: 0.3207, Train Acc: 95.82% | Val Loss: 0.4231, Val Acc: 92.63%


Epoch [8/10] - Train Loss: 0.2763, Train Acc: 96.64% | Val Loss: 0.3970, Val Acc: 92.17%


Epoch [9/10] - Train Loss: 0.2422, Train Acc: 97.12% | Val Loss: 0.3748, Val Acc: 92.57%


Epoch [10/10] - Train Loss: 0.2147, Train Acc: 97.42% | Val Loss: 0.3619, Val Acc: 92.51%
Finished ResNet-18 (Transfer Learning). Best Validation Loss: 0.3619
